# 3. Anonymization

Three passes on every title and text (see `anonymization_ner.py`): links and e-mail addresses → `[URL]`, `[EMAIL]`;
the outlet's own name written from its domain → `[OUTLET]`; then a standard named-entity recognizer, GLiNER (zero-shot
NER, `urchade/gliner_large-v2.1`), with the labels "news outlet" → `[OUTLET]` and "journalist" → `[AUTHOR]` (every
outlet and every journalist, not only the publisher). Nothing else changes, no rewriting.
Check: articles that still contain the name of their own outlet.

In [ ]:
THRESHOLD = 0.5

In [ ]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())
import anonymization_ner as AX

AX.THRESHOLD = THRESHOLD
art = pd.read_csv(C.OUT / "articles.csv.gz")
art["title"] = art.title.fillna(""); art["text"] = art.text.fillna("")

## Passes 1-2: links, e-mails, the outlet's own name

In [ ]:
ruled, counts = AX.rules(art)
print("replacements:", counts)

## Pass 3: GLiNER (news outlets and journalists)

In [ ]:
spans = AX.run(ruled, LOG)
print(f"spans: {len(spans):,} in {spans.article_id.nunique():,} articles ({spans.article_id.nunique() / len(art):.1%})")
text = {(a, f): t for f in ["title", "text"] for a, t in zip(ruled.article_id, ruled[f])}
spans["span"] = [text[(a, f)][s:e] for a, f, s, e in zip(spans.article_id, spans.field, spans.start, spans.end)]
for label, g in spans.groupby("label"):
    print(f"{label}: {len(g):,}; most frequent", g.span.value_counts().head(15).index.tolist())

In [ ]:
anon = AX.apply(ruled, spans)
before, after = AX.own_name_left(art), AX.own_name_left(anon)
print(f"articles with the name of their own outlet: before {before['own name left'].mean():.1%}, after {after['own name left'].mean():.1%} "
      f"({after['own name left'].sum():,} articles)")
print("outlets with the most leftovers:", after.groupby("domain")["own name left"].sum().sort_values(ascending=False).head(10).to_dict())
anon[["article_id", "domain", "title", "date", "text", "score", "label", "level"]].to_csv(C.OUT / "articles_anonymous.csv.gz", index=False)
after.to_csv(C.OUT / "anonymization_check.csv", index=False)
print("saved", C.OUT / "articles_anonymous.csv.gz")